In [4]:
from pathlib import Path

import pandas as pd

hist_file = Path("data") / "historico.parquet"
df_hist = pd.read_parquet(hist_file)

In [ ]:
scout_cols = [c for c in df_hist.columns if c.startswith("scout_")]
df_hist[scout_cols] = df_hist[scout_cols].fillna(0)
df_hist = df_hist.sort_values(["atleta_id", "rodada"]).copy()

for janela in [3, 5, 10]:
    df_hist[f"media_pts_{janela}r"] = (
        df_hist.groupby("atleta_id")["pontos"]
        .transform(lambda x: x.shift(1).rolling(janela, min_periods=1).mean())
    )
    df_hist[f"std_pts_{janela}r"] = (
        df_hist.groupby("atleta_id")["pontos"]
        .transform(lambda x: x.shift(1).rolling(janela, min_periods=1).std().fillna(0))
    )

df_hist["pts_ultima_rodada"] = df_hist.groupby("atleta_id")["pontos"].shift(1)
df_hist["tendencia"] = df_hist["media_pts_3r"] - df_hist["media_pts_10r"]

for col in ["scout_G", "scout_A", "scout_SG", "scout_GS", "scout_DD"]:
    if col in df_hist.columns:
        df_hist[f"acc_{col}"] = (
            df_hist.groupby("atleta_id")[col]
            .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
        )

df_hist["preco_lag1"] = df_hist.groupby("atleta_id")["preco"].shift(1)

df_hist["jogou"] = (df_hist["pontos"] > 0).astype(int)
df_hist["regularidade_5r"] = (
    df_hist.groupby("atleta_id")["jogou"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

,rodada,atleta_id,apelido,posicao_id,clube_id,clube_nome,status_id,pontos,preco,media,...,std_pts_10r,pts_ultima_rodada,tendencia,acc_scout_G,acc_scout_A,acc_scout_SG,acc_scout_GS,preco_lag1,jogou,regularidade_5r
752,5,82627,Luiz Araújo,5,262,FLA,None,0.0,0.0,0.0,...,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN
753,6,82627,Luiz Araújo,5,262,FLA,None,-0.3,0.0,0.0,...,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0,0.000000
754,7,82627,Luiz Araújo,5,262,FLA,None,17.6,0.0,0.0,...,0.212132,-0.3,0.000000,0.0,0.0,0.0,0.0,0.0,1,0.000000
755,9,82627,Luiz Araújo,5,262,FLA,None,1.7,0.0,0.0,...,10.249065,17.6,0.000000,1.0,1.0,0.0,0.0,0.0,1,0.333333
756,10,82627,Luiz Araújo,5,262,FLA,None,3.0,0.0,0.0,...,8.611814,1.7,1.583333,1.0,1.0,0.0,0.0,0.0,1,0.500000
757,11,82627,Luiz Araújo,5,262,FLA,None,0.5,0.0,0.0,...,7.499000,3.0,3.033333,1.0,1.0,0.0,0.0,0.0,1,0.600000
758,12,82627,Luiz Araújo,5,262,FLA,None,0.0,0.0,0.0,...,6.893693,0.5,-2.016667,1.0,1.0,0.0,0.0,0.0,0,0.800000
759,13,82627,Luiz Araújo,5,262,FLA,None,0.0,0.0,0.0,...,6.450692,0.0,-2.047619,1.0,1.0,0.0,0.0,0.0,0,0.800000
760,14,82627,Luiz Araújo,5,262,FLA,None,0.9,0.0,0.0,...,6.079341,0.0,-2.645833,0.0,0.0,0.0,0.0,0.0,1,0.600000
971,1,86757,Everton,5,262,FLA,None,3.2,0.0,0.0,...,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN


In [ ]:
import importlib, cartola_team_builder
importlib.reload(cartola_team_builder)
from cartola_team_builder import comparar_jogadores

In [ ]:
from cartola_team_builder import (
    construir_features, treinar_modelo
)
from cartola_collector import DATA_DIR
import pandas as pd

df_hist = pd.read_parquet(DATA_DIR / "historico.parquet")
df_partidas = pd.read_parquet(DATA_DIR / "partidas.parquet")
df_odds = pd.read_parquet(DATA_DIR / "odds.parquet")

rodada_alvo = 13  # ajuste

df_feat = construir_features(
    df_hist[df_hist["rodada"] < rodada_alvo], df_partidas, df_odds
)
model, feat_cols = treinar_modelo(df_feat, rodada_corte=rodada_alvo)

# Now pass df_feat (not df_mercado) — comparar_jogadores picks the latest row per player
comparar_jogadores(model, df_feat, feat_cols, ["Pedro", "Samuel Lino"], top_n_features=12)

2026-05-11 16:47:04,008 WARNING 'Pedro' casou 64 jogadores; usando o primeiro: Pedro Henrique
2026-05-11 16:47:04,011 WARNING 'Samuel Lino' casou 10 jogadores; usando o primeiro: Samuel Lino



COMPARAÇÃO: Pedro Henrique vs Samuel Lino
  Pedro Henrique:  raw_pred = 2.856  (bias = 2.720)
  Samuel Lino:  raw_pred = 2.430  (bias = 2.720)
----------------------------------------------------------------------
Top 12 features por |Δ shap|:
         feature  value_Pedro Henrique  shap_Pedro Henrique  value_Samuel Lino  shap_Samuel Lino  shap_delta
     posicao_enc                 3.000               -0.097              5.000            -0.275       0.178
   oppo_def_avg5                 0.000                0.384              0.000             0.283       0.101
       clube_enc                10.000                0.207              1.000             0.155       0.052
           mando                -1.000               -0.034             -1.000            -0.072       0.037
   oppo_off_avg5                 0.000                0.025              0.000            -0.000       0.025
      std_pts_3r                 0.000               -0.010              0.000            -0.035     

,feature,value_Pedro Henrique,shap_Pedro Henrique,value_Samuel Lino,shap_Samuel Lino,shap_delta
0,posicao_enc,3.00,-0.097185,5.00,-0.275355,0.178170
1,oppo_def_avg5,0.00,0.384159,0.00,0.282924,0.101236
2,clube_enc,10.00,0.206897,1.00,0.154596,0.052301
3,mando,-1.00,-0.034462,-1.00,-0.071779,0.037318
4,oppo_off_avg5,0.00,0.025141,0.00,-0.000093,0.025235
5,std_pts_3r,0.00,-0.009582,0.00,-0.034571,0.024988
6,std_pts_5r,0.00,0.018577,0.00,-0.000675,0.019252
7,prob_draw,0.34,0.073441,0.34,0.055231,0.018210
8,oppo_def_x_mando,0.00,0.006596,0.00,0.021714,-0.015118
9,media_pts_3r,0.00,-0.062126,0.00,-0.049587,-0.012540
